# Image + box → LiteMedSAM → crops → Darma scores

Read both model lessons before executing. This is a software integration demonstration with an untrained classification head. No class names or diagnostic labels are assigned. Select a GPU runtime. The official LiteMedSAM source is unchanged.

## 1. Source and GPU setup

In [ ]:
import sys, subprocess, json
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU."
DEVICE="cuda:0"
print("GPU:",torch.cuda.get_device_name(0),"PyTorch:",torch.__version__)
REPO=Path("/content/MedSAM_student")
if not REPO.exists():
    subprocess.run(["git","clone","--branch","codex/student-foundation","--single-branch",
                    "https://github.com/KrishnachaithanyaThummala/MedSAM.git",str(REPO)],check=True)
PROJECT=REPO/"student_project"
sys.path.insert(0,str(PROJECT))
MED=Path("/content/MedNet_student")
if not MED.exists(): subprocess.run(["git","clone","https://github.com/Md-Ferdous/MedNet.git",str(MED)],check=True)
subprocess.run(["git","checkout","--detach","6adb072630811b1d39568a87732aed96dc39c9ae"],cwd=MED,check=True)
print("Project commit:",subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip())


**Result/inference:** Source commits and GPU identify the runtime. Project additions live under student_project; original model files remain in the parent checkout.

## 2. Install inference-only dependencies

In [ ]:
subprocess.run([sys.executable,"-m","pip","install","timm==0.9.12","gdown==5.2.0","opencv-python-headless","matplotlib","pandas","tqdm"],check=True)
subprocess.run([sys.executable,"-c","from tiny_vit_sam import TinyViT; from segment_anything.modeling import PromptEncoder,MaskDecoder"],cwd=REPO,check=True)


**Result/inference:** Successful imports establish dependency availability. If Colab requests a restart, restart and rerun setup first.

## 3. Official checkpoint and ultrasound example

In [ ]:
import gdown,hashlib,numpy as np
CHECKPOINT=REPO/"work_dir/LiteMedSAM/lite_medsam.pth"
SAMPLE=PROJECT/"ultrasound_demo.npz"
if not CHECKPOINT.exists():gdown.download(id="18Zed-TUTsmr2zc5CHUWd5Tu13nb6vq6z",output=str(CHECKPOINT),quiet=False)
assert hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()=="79d8c9dca6db4d69d3f905579e5250af05e859fff9c1f543e89a513c3028ce76"
if not SAMPLE.exists():gdown.download(id="1SBsOOXOCavdQbVLb82O0R0vIlS5TqIKg",output=str(SAMPLE),quiet=False)
with np.load(SAMPLE,allow_pickle=False) as data:image,box=data["imgs"],data["boxes"][0]
print(image.shape,box)


**Result/inference:** The box comes from the authors' supplied demo. For future research images the box must be provided by a human/annotation; autonomous detection is not implemented.

## 4. Execute the modular pipeline

In [ ]:
from src.pipeline import execute
OUT=PROJECT/"outputs/integrated_gpu"
report=execute(image,box,REPO,MED,CHECKPOINT,OUT,DEVICE)
print(json.dumps(report,indent=2))
assert len(report["crops"])==3
assert all(c["output_shape"]==[1,2] for c in report["crops"])


**Result/inference:** The output records tight, 10% and 20% crops, each with two untrained numerical scores. Margins expand each side relative to tight width/height and are clipped to image bounds. Empty masks produce a clear error.

## 5. Inspect actual saved images and crops

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
fig,axes=plt.subplots(1,5,figsize=(18,5))
paths=[OUT/"image.png",OUT/"mask.png"]+[OUT/"crops"/c["file"] for c in report["crops"]]
for ax,p in zip(axes,paths):ax.imshow(Image.open(p));ax.set_title(p.name);ax.axis("off")
plt.tight_layout();plt.show()


**Result/inference:** Verify the crop covers the foreground and larger margins add context. This visual check is distinct from mask accuracy evaluation.

## 6. Tests and saved deliverables

In [ ]:
subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"],cwd=PROJECT,check=True)
import shutil
from google.colab import files
files.download(shutil.make_archive("/content/LiteMedSAM_MedNet_results","zip",OUT))


**Result/inference:** Keep the ZIP and save the notebook with outputs. Record Colab success only after execution. Timing includes loading/I/O and is not a pure GPU benchmark.